# Recursive Chunking — Multi-PDF Experiment

This notebook evaluates **Recursive Character Text Splitting** across a whole folder of PDFs.

To add a new document: drop the PDF into `data/recursive_friendly/`, add its questions to
`data/ground_truth.json`, and re-run. Nothing in the code is tied to a specific file.
(The Markdown pipeline has been removed on purpose.)

Pipeline:

PDFs → Extract (per page) → Normalize → Chunk (per chunk size) → Embed →
Retrieve (one shared index over all PDFs) → Evaluate

**How retrieval is evaluated**

All PDFs are chunked and indexed *together*, so a question about asthma has to beat the
chunks of every other document. A retrieved chunk counts as **relevant** only if it comes
from the PDF the question belongs to *and* contains at least one of the question's
evidence phrases.

| Metric | Meaning |
|---|---|
| Precision@k | relevant chunks in the top k ÷ k |
| Recall@k | share of the question's evidence phrases found in the top k chunks |
| Hit rate@k | 1 if any top-k chunk is relevant, else 0 |
| MRR@k | 1 ÷ rank of the first relevant chunk (0 if none in the top k) |

Every metric is averaged over all questions.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf  # PyMuPDF
import matplotlib.pyplot as plt

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

## Configuration

Everything you are likely to tune lives in this one cell.

In [ ]:
BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / "data" / "recursive_friendly"
GROUND_TRUTH_PATH = BASE_DIR / "data" / "ground_truth.json"

# Chunking
CHUNK_SIZES = [200, 300, 400, 500, 700]
CHUNK_OVERLAP = 50
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

# Evaluation
K_VALUES = (1, 3, 5)
TOP_K = max(K_VALUES)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("PDF folder:", PDF_DIR)
print("Ground truth:", GROUND_TRUTH_PATH)

## 1. Document Ingestion

Each PDF is read page by page. Every page becomes a LangChain `Document` with its
source filename and page number, so retrieved chunks can be traced back to their
original location.

In [ ]:
def load_pdf(path):
    """Load a PDF page by page. Each page becomes a LangChain Document."""
    documents = []

    with pymupdf.open(path) as pdf:
        total_pages = len(pdf)

        for page_number, page in enumerate(pdf, start=1):
            documents.append(
                Document(
                    page_content=page.get_text("text"),
                    metadata={
                        "source": path.name,
                        "file_type": "pdf",
                        "page": page_number,
                        "total_pages": total_pages,
                    },
                )
            )

    return documents


def load_all_pdfs(pdf_dir):
    """Load every PDF in a folder (the .pdf extension is matched case-insensitively)."""
    assert pdf_dir.exists(), f"PDF folder not found: {pdf_dir}"

    pdf_paths = sorted(p for p in pdf_dir.iterdir() if p.suffix.lower() == ".pdf")
    assert pdf_paths, f"No PDFs found in {pdf_dir}"

    documents = []
    for path in pdf_paths:
        documents.extend(load_pdf(path))

    return pdf_paths, documents


pdf_paths, documents = load_all_pdfs(PDF_DIR)

print(f"PDFs found: {len(pdf_paths)}")
for path in pdf_paths:
    print(" -", path.name)
print("Total pages:", len(documents))

### Text Normalization

Normalization is conservative: it removes extraction noise without destroying
document structure (Windows line endings, trailing whitespace, excessive blank lines).

In [ ]:
def normalize_text(text):
    """Perform conservative text normalization."""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove trailing whitespace from each line
    text = "\n".join(line.rstrip() for line in text.split("\n"))

    # Reduce excessive blank lines while preserving paragraph separation
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


for document in documents:
    document.page_content = normalize_text(document.page_content)

### Ingestion Summary

One row per PDF. Pages with no extractable text are dropped and counted.
A PDF where most pages are empty is usually a **scan** and needs OCR before it is useful.

In [ ]:
page_report = pd.DataFrame([
    {
        "source": d.metadata["source"],
        "page": d.metadata["page"],
        "chars": len(d.page_content),
    }
    for d in documents
])
page_report["blank"] = page_report["chars"] == 0

ingestion_summary = (
    page_report
    .groupby("source")
    .agg(
        pages=("page", "count"),
        blank_pages=("blank", "sum"),
        total_chars=("chars", "sum"),
        avg_chars_per_page=("chars", "mean"),
    )
    .round(0)
    .astype(int)
)

scanned_suspects = ingestion_summary[
    ingestion_summary["blank_pages"] >= 0.5 * ingestion_summary["pages"]
]
if not scanned_suspects.empty:
    print("WARNING - mostly empty after extraction (scanned PDFs? they need OCR):")
    print(scanned_suspects.index.tolist())

# Drop pages with no text
documents = [d for d in documents if d.page_content]

ingestion_summary

### Ingestion Validation

In [ ]:
def validate_documents(documents, pdf_paths):
    """Basic validation for ingested documents."""

    assert documents, "No text could be extracted from any PDF."

    sources_with_text = {d.metadata["source"] for d in documents}
    no_text = [p.name for p in pdf_paths if p.name not in sources_with_text]
    assert not no_text, f"No extractable text in: {no_text}"

    for document in documents:
        metadata = document.metadata

        assert document.page_content.strip(), f"Empty document: {metadata}"

        for key in ("source", "file_type", "page"):
            assert key in metadata, f"Missing '{key}' metadata: {metadata}"

    print("All ingestion validation checks passed.")


validate_documents(documents, pdf_paths)

## 2. Recursive Chunking

The main experimental variable is `chunk_size`. Overlap and separators stay fixed, so
any difference in results comes from chunk size alone.

All chunk sizes are built in one loop over **all PDFs**. Chunks never cross page
boundaries because splitting happens per page.

In [ ]:
def create_recursive_splitter(chunk_size, chunk_overlap=CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=SEPARATORS,
    )


pdf_chunks_by_size = {}

for size in CHUNK_SIZES:
    chunks = create_recursive_splitter(size).split_documents(documents)

    for chunk_id, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = chunk_id

    pdf_chunks_by_size[size] = chunks

for size, chunks in pdf_chunks_by_size.items():
    print(f"Chunk size {size}: {len(chunks)} chunks")

In [ ]:
def preview_chunks(chunk_size, n=3, source=None):
    """Print the first n chunks for a chunk size, optionally for one PDF only."""
    chunks = pdf_chunks_by_size[chunk_size]

    if source:
        chunks = [c for c in chunks if c.metadata["source"] == source]

    for i, chunk in enumerate(chunks[:n], start=1):
        meta = chunk.metadata
        print(f"\n--- Chunk {i} | {meta['source']} p.{meta['page']} | {len(chunk.page_content)} chars ---")
        print(chunk.page_content)


preview_chunks(200)

### Chunk statistics

In [ ]:
def chunk_statistics(chunks):
    lengths = [len(chunk.page_content) for chunk in chunks]

    return {
        "num_chunks": len(chunks),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "avg_chars": np.mean(lengths),
        "median_chars": np.median(lengths),
    }


stats_table = pd.DataFrame([
    {"chunk_size": size, **chunk_statistics(chunks)}
    for size, chunks in pdf_chunks_by_size.items()
]).round(1)

stats_table

Chunks per PDF (rows) for each chunk size (columns):

In [ ]:
chunks_per_pdf = pd.DataFrame({
    size: Counter(c.metadata["source"] for c in chunks)
    for size, chunks in pdf_chunks_by_size.items()
}).fillna(0).astype(int)

chunks_per_pdf.index.name = "source"
chunks_per_pdf.columns.name = "chunk_size"
chunks_per_pdf.loc["TOTAL"] = chunks_per_pdf.sum()

chunks_per_pdf

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(stats_table["chunk_size"], stats_table["num_chunks"], marker="o")
axes[0].set_xlabel("Chunk Size (characters)")
axes[0].set_ylabel("Number of Chunks")
axes[0].set_title("Number of chunks (all PDFs)")
axes[0].grid(True)

axes[1].plot(stats_table["chunk_size"], stats_table["avg_chars"], marker="o")
axes[1].set_xlabel("Chunk Size (characters)")
axes[1].set_ylabel("Average Chunk Length (characters)")
axes[1].set_title("Average chunk length (all PDFs)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Embedding Model

`all-MiniLM-L6-v2` silently truncates input beyond its token limit, so the table below
shows how many chunks of each size would be cut off.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(embedding_model)

In [ ]:
tokenizer = embedding_model.tokenizer
model_limit = embedding_model.max_seq_length  # 256 for all-MiniLM-L6-v2

token_lengths_by_size = {
    size: [
        len(tokenizer.encode(chunk.page_content, add_special_tokens=False))
        for chunk in chunks
    ]
    for size, chunks in pdf_chunks_by_size.items()
}

fig, axes = plt.subplots(1, len(CHUNK_SIZES), figsize=(4 * len(CHUNK_SIZES), 3.5))
for ax, (size, lengths) in zip(np.atleast_1d(axes), token_lengths_by_size.items()):
    ax.hist(lengths, bins=20)
    ax.axvline(model_limit, color="red", linestyle="--", label=f"limit ({model_limit})")
    ax.set_title(f"{size} chars")
    ax.set_xlabel("Tokens per chunk")
np.atleast_1d(axes)[0].set_ylabel("Number of chunks")
np.atleast_1d(axes)[0].legend()
plt.tight_layout()
plt.show()

pd.DataFrame([
    {
        "chunk_size": size,
        "max_tokens": max(lengths),
        "pct_over_model_limit": round(100 * np.mean(np.array(lengths) > model_limit), 2),
    }
    for size, lengths in token_lengths_by_size.items()
])

In [ ]:
pdf_embeddings_by_size = {}

for size, chunks in pdf_chunks_by_size.items():
    pdf_embeddings_by_size[size] = embedding_model.encode(
        [chunk.page_content for chunk in chunks],
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True,
    )

    print(f"Chunk size {size}: {pdf_embeddings_by_size[size].shape}")

### Retrieval

The chunk embeddings are L2-normalized, so the dot product is the cosine similarity.
Retrieval runs over the chunks of **all** PDFs at once.

In [ ]:
def rank_chunks(query_embedding, chunks, chunk_embeddings, top_k=TOP_K):
    """Return the top_k chunks for an already-embedded query."""
    similarities = chunk_embeddings @ query_embedding
    ranked_indices = np.argsort(similarities)[::-1][:top_k]

    return [
        {
            "rank": rank,
            "chunk_index": int(index),
            "score": float(similarities[index]),
            "chunk": chunks[index],
        }
        for rank, index in enumerate(ranked_indices, start=1)
    ]


def retrieve_chunks(query, chunks, chunk_embeddings, top_k=TOP_K):
    """Embed a text query and return the top_k chunks."""
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    return rank_chunks(query_embedding, chunks, chunk_embeddings, top_k)

## 4. Ground Truth

Questions live in `data/ground_truth.json`, one object per question:

```json
{
  "id": "diabetes_Q1",
  "source": "diabetes_guideline.pdf",
  "type": "fact",
  "question": "How is type 2 diabetes diagnosed?",
  "evidence": ["fasting plasma glucose", "HbA1c of 6.5%"]
}
```

- `id` must be unique across all PDFs.
- `source` is the filename of the PDF that contains the answer.
- `evidence` phrases are copied from that PDF. Case, whitespace, curly quotes and
  ligatures are ignored when matching. Keep them short so each phrase fits inside a single chunk.
- `type` is optional (for example `fact`, `paraphrase`, `multi`).

Validation checks every phrase against the **individual pages** of its PDF, because
chunks never cross page boundaries, so a phrase spanning two pages can never be retrieved.

In [ ]:
def norm(text):
    """Normalize text for evidence matching."""
    text = unicodedata.normalize("NFKC", text)
    text = (
        text.replace("\u2019", "'").replace("\u2018", "'")
            .replace("\u201c", '"').replace("\u201d", '"')
            .replace("\u2013", "-").replace("\u2014", "-")
    )
    return re.sub(r"\s+", " ", text).lower().strip()


with open(GROUND_TRUTH_PATH, encoding="utf-8") as f:
    eval_questions = json.load(f)

print("Questions loaded:", len(eval_questions))

### Finding evidence phrases

Evidence must be copied from the *extracted* text, which can differ from what you see in a
PDF viewer (line breaks, hyphenation, table layout). `find_evidence` searches the extracted
pages and prints verbatim snippets with page numbers that you can paste into `ground_truth.json`.

In [ ]:
def find_evidence(keyword, source=None, context=80, max_hits=10):
    """Search the extracted pages for a keyword and print copy-pasteable snippets."""
    hits = 0

    for d in documents:
        if source and d.metadata["source"] != source:
            continue

        text = re.sub(r"\s+", " ", d.page_content)

        for match in re.finditer(re.escape(keyword), text, flags=re.IGNORECASE):
            start = max(0, match.start() - context)
            end = min(len(text), match.end() + context)

            print(f"[{d.metadata['source']} | p.{d.metadata['page']}]  ...{text[start:end]}...")
            hits += 1

            if hits >= max_hits:
                print(f"(stopped after {max_hits} hits)")
                return

    if hits == 0:
        print(f"No matches for {keyword!r}. Try a shorter keyword or a different spelling.")


# Example:
# find_evidence("systolic blood pressure", source="who_hypertension_pharmacological_treatment_2021.pdf")

In [ ]:
def validate_ground_truth(questions, documents, max_phrase_chars):
    """Check ids, sources and evidence phrases before spending time on evaluation."""

    page_texts = {}
    for d in documents:
        page_texts.setdefault(d.metadata["source"], []).append(norm(d.page_content))

    problems, warnings, seen_ids = [], [], set()

    for q in questions:
        qid = q.get("id", "<no id>")

        for field in ("id", "source", "question", "evidence"):
            if not q.get(field):
                problems.append(f"{qid}: missing or empty '{field}'")

        if qid in seen_ids:
            problems.append(f"{qid}: duplicate id")
        seen_ids.add(qid)

        source = q.get("source")
        if source not in page_texts:
            problems.append(f"{qid}: source '{source}' is not one of the loaded PDFs")
            continue

        for phrase in q.get("evidence", []):
            if not any(norm(phrase) in page for page in page_texts[source]):
                problems.append(f"{qid}: evidence not found on any single page of {source} -> {phrase!r}")
            if len(phrase) > max_phrase_chars:
                warnings.append(f"{qid}: evidence is {len(phrase)} chars (> {max_phrase_chars}); may not fit in one chunk -> {phrase[:50]!r}...")

    for w in warnings:
        print("WARNING:", w)
    for p in problems:
        print("ERROR:  ", p)
    assert not problems, f"{len(problems)} ground-truth problem(s) - fix them and re-run."

    questions_per_pdf = Counter(q["source"] for q in questions)
    print("Ground truth OK. Questions per PDF:")
    for name in sorted(page_texts):
        count = questions_per_pdf.get(name, 0)
        note = "" if count else "  (no questions - acts only as a distractor document)"
        print(f"  {name}: {count}{note}")


validate_ground_truth(
    eval_questions,
    documents,
    max_phrase_chars=min(CHUNK_SIZES) - CHUNK_OVERLAP,
)

## 5. Retrieval Evaluation

For every chunk size and every question, the top 5 chunks are retrieved from the shared
index, and Precision, Recall, Hit rate and MRR are computed at k = 1, 3 and 5.

In [ ]:
def score_ranked_list(evidence, source, ranked, ks):
    """
    Score one question.

    ranked: list of (normalized_chunk_text, chunk_source) in rank order.
    Returns {k: {precision, recall, hit_rate, mrr}}.
    """
    phrases = [norm(e) for e in evidence]

    is_relevant = [
        chunk_source == source and any(p in text for p in phrases)
        for text, chunk_source in ranked
    ]

    scores = {}
    for k in ks:
        window = ranked[:k]
        flags = is_relevant[:k]

        # Evidence recall only counts chunks from the correct PDF
        window_texts = [text for text, chunk_source in window if chunk_source == source]
        recall = sum(any(p in t for t in window_texts) for p in phrases) / len(phrases)

        first_relevant_rank = next((r for r, f in enumerate(flags, start=1) if f), None)

        scores[k] = {
            "precision": sum(flags) / k,
            "recall": recall,
            "hit_rate": float(any(flags)),
            "mrr": 1 / first_relevant_rank if first_relevant_rank else 0.0,
        }

    return scores

In [ ]:
# Pre-compute normalized chunk text once per chunk size
norm_texts_by_size = {
    size: [norm(c.page_content) for c in chunks]
    for size, chunks in pdf_chunks_by_size.items()
}

# Embed all questions in one batch
question_embeddings = embedding_model.encode(
    [q["question"] for q in eval_questions],
    normalize_embeddings=True,
)

eval_rows = []

for size, chunks in pdf_chunks_by_size.items():
    for q, q_embedding in zip(eval_questions, question_embeddings):

        results = rank_chunks(q_embedding, chunks, pdf_embeddings_by_size[size], TOP_K)

        ranked = [
            (norm_texts_by_size[size][r["chunk_index"]], chunks[r["chunk_index"]].metadata["source"])
            for r in results
        ]

        for k, scores in score_ranked_list(q["evidence"], q["source"], ranked, K_VALUES).items():
            eval_rows.append({
                "chunk_size": size,
                "k": k,
                "id": q["id"],
                "source": q["source"],
                "type": q.get("type", "unlabeled"),
                **scores,
            })

eval_df = pd.DataFrame(eval_rows)
print(f"{len(eval_questions)} questions x {len(CHUNK_SIZES)} chunk sizes x {len(K_VALUES)} k values = {len(eval_df)} rows")

### Complete results table

Mean Precision, Recall, Hit rate and MRR for every chunk size and every k.

In [ ]:
METRICS = ["precision", "recall", "hit_rate", "mrr"]

summary = eval_df.groupby(["chunk_size", "k"])[METRICS].mean().round(3)
summary

Same numbers, wide format (one column per metric and k):

In [ ]:
summary.unstack("k")

### Best chunk size per metric

For each k, the chunk size with the highest score (ties go to the smaller size).

In [ ]:
best_chunk_size = pd.DataFrame({
    metric: summary[metric].unstack("k").idxmax()
    for metric in METRICS
})
best_chunk_size.index.name = "k"

best_chunk_size

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4))

for ax, metric in zip(axes, METRICS):
    for k in K_VALUES:
        series = summary.xs(k, level="k")[metric]
        ax.plot(series.index, series.values, marker="o", label=f"k={k}")

    ax.set_title(metric)
    ax.set_xlabel("Chunk Size (characters)")
    ax.set_ylim(0, 1.05)
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

### Per-PDF breakdown (k = 5)

Shows whether one document is dragging the average down.

In [ ]:
per_pdf = (
    eval_df[eval_df["k"] == TOP_K]
    .groupby(["source", "chunk_size"])
    .agg(
        questions=("id", "nunique"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        hit_rate=("hit_rate", "mean"),
        mrr=("mrr", "mean"),
    )
    .round(3)
)

per_pdf

### Questions that failed

Questions with no relevant chunk in the top 5. Use `show_retrieval_results` below to see why.

In [ ]:
misses = (
    eval_df[(eval_df["k"] == TOP_K) & (eval_df["hit_rate"] == 0)]
    [["id", "source", "type", "chunk_size"]]
    .sort_values(["id", "chunk_size"])
    .reset_index(drop=True)
)

print(f"{len(misses)} misses at k={TOP_K}")
misses

In [ ]:
def show_retrieval_results(question_id, chunk_size, top_k=TOP_K):
    """Print the top-k chunks for one question and mark which ones are relevant."""

    q = next(q for q in eval_questions if q["id"] == question_id)
    chunks = pdf_chunks_by_size[chunk_size]

    results = retrieve_chunks(
        q["question"], chunks, pdf_embeddings_by_size[chunk_size], top_k=top_k
    )

    phrases = [norm(e) for e in q["evidence"]]

    print("=" * 80)
    print(f"{question_id}: {q['question']}")
    print(f"Expected source: {q['source']} | Chunk size: {chunk_size}")
    print("=" * 80)

    for r in results:
        chunk = r["chunk"]
        meta = chunk.metadata
        relevant = (
            meta["source"] == q["source"]
            and any(p in norm(chunk.page_content) for p in phrases)
        )

        print(
            f"\nRank {r['rank']} | {meta['source']} p.{meta['page']} | "
            f"chunk {r['chunk_index']} | sim {r['score']:.4f} | "
            f"{'RELEVANT' if relevant else '-'}"
        )
        print("-" * 80)
        print(chunk.page_content)


show_retrieval_results(eval_questions[0]["id"], CHUNK_SIZES[0])